## Normalisasi tanggal dan penanda soal korpus kalimat


## 1. Fungsi normalisasi tanggal

In [ ]:
import re
from datetime import date

NAMA_BULAN = [
    "januari", "februari", "maret", "april",
    "mei", "juni", "juli", "agustus",
    "september", "oktober", "november", "desember"
]

VARIASI_BULAN = {
    "jan": 1, "januari": 1, "january": 1, "jnuari": 1, "jnuary": 1,
    "peb": 2, "pebruari": 2,
    "feb": 2, "februari": 2, "february": 2,
    "mar": 3, "mrt": 3, "mart": 3, "marett": 3,
    "maart": 3, "maret": 3, "march": 3, "mrch": 3,
    "apr": 4, "april": 4, "apirl": 4, "aprill": 4,
    "mei": 5, "may": 5, "mey": 5, "mai": 5,
    "jun": 6, "juni": 6, "djuni": 6, "june": 6, "juney": 6,
    "jul": 7, "juli": 7, "djuli": 7, "july": 7, "jully": 7,
    "agu": 8, "agt": 8, "ags": 8, "agst": 8,
    "agus": 8, "agustus": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "okt": 10, "oct": 10, "oktober": 10, "october": 10,
    "nop": 11, "nov": 11, "nopember": 11, "november": 11,
    "des": 12, "dec": 12, "desember": 12, "december": 12,
}

TAHUN = r"(?:1[5-9]\d\d|20\d\d)"
HARI = r"\d{1,2}"

BULAN = (
    "(?:"
    + "|".join(
        re.escape(b)
        for b in sorted(VARIASI_BULAN, key=len, reverse=True)
    )
    + ")"
)

# Melindungi rentang, contoh 10-17 Juli 1945 atau 28 Februari–1 Maret 1945
RENTANG = (
    HARI
    + rf"(?:\s+{BULAN}\.?(?:\s+{TAHUN})?)?"
    + r"\s*(?:-|–|—|sampai|hingga|s\.d\.)\s*"
    + HARI
    + rf"\s+{BULAN}\.?(?:\s+{TAHUN})?"
)

POLA_TANGGAL = re.compile(
    r"(?<![\w./-])(?:"
    + rf"(?P<rentang>{RENTANG})"
    + rf"|(?P<iso>{TAHUN}-\d{{2}}-\d{{2}})"
    + rf"|(?P<angka>{HARI}(?P<pemisah>[./-])"
      rf"{HARI}(?P=pemisah){TAHUN})"
    + rf"|(?P<lengkap>{HARI}\s+{BULAN}\.?\s+{TAHUN})"
    + rf"|(?P<tanpa_tahun>{HARI}\s+{BULAN})"
    + r")(?![\w/-]|\.\d)",
    re.IGNORECASE,
)


def normalisasi_tanggal(teks):
    def ganti(cocok):
        asli = cocok.group()

        if cocok.group("rentang"):
            return asli

        try:
            if cocok.group("iso"):
                tahun, bulan, hari = map(int, asli.split("-"))

            elif cocok.group("angka"):
                hari, bulan, tahun = map(
                    int, re.split(r"[./-]", asli)
                )

            else:
                bagian = asli.split()
                hari = int(bagian[0])
                bulan = VARIASI_BULAN[
                    bagian[1].lower().rstrip(".")
                ]
                tahun = int(bagian[2]) if len(bagian) == 3 else None
            date(
                tahun if tahun is not None else 2000,
                bulan,
                hari,
            )

            hasil = f"{hari} {NAMA_BULAN[bulan - 1]}"

            if tahun is not None:
                hasil += f" {tahun}"

            return hasil

        except (ValueError, KeyError):
            return asli

    return POLA_TANGGAL.sub(ganti, teks)

## 3. Unggah korpus kalimat


In [ ]:
import json
import re
from copy import deepcopy
from pathlib import Path
from google.colab import files

unggahan = files.upload()

data = json.loads(
    unggahan[nama_input].decode("utf-8-sig")
)

ALIAS_PETA = data["metadata"]["aturan_alias"]

daftar_alias = sorted(ALIAS_PETA, key=len, reverse=True)

POLA_ALIAS = re.compile(
    r"(?<!\w)(?:"
    + "|".join(re.escape(nama) for nama in daftar_alias)
    + r")(?!\w)"
)


def normalisasi_alias(teks):
    return POLA_ALIAS.sub(
        lambda cocok: ALIAS_PETA[cocok.group()],
        teks,
    )


def normalisasi_pencarian(teks):
    teks = re.sub(r"\s+", " ", teks).strip().lower()
    teks = normalisasi_alias(teks)
    return normalisasi_tanggal(teks)


kalimat = data["kalimat"]

print("Jumlah kalimat:", len(kalimat))
print("Jumlah aturan alias:", len(ALIAS_PETA))

Saving korpus_kalimat_alias.json to korpus_kalimat_alias (1).json
File: korpus_kalimat_alias (1).json
Jumlah kalimat: 3140
Jumlah aturan alias: 17


## 5. Terapkan tanggal dan tandai soal

In [ ]:
asli = deepcopy(kalimat)
berubah = []

for k in kalimat:
    baru = normalisasi_tanggal(k["teks_cari"])

    if baru != k["teks_cari"]:
        berubah.append(k["id_kalimat"])

    k["teks_cari"] = baru

assert len(kalimat) == len(asli)

for sebelum, sesudah in zip(asli, kalimat):
    assert {
        key: nilai
        for key, nilai in sebelum.items()
        if key != "teks_cari"
    } == {
        key: nilai
        for key, nilai in sesudah.items()
        if key != "teks_cari"
    }, f"Field lain berubah: {sebelum['id_kalimat']}"

print("Jumlah kalimat:", len(kalimat))
print("Kalimat yang berubah karena tanggal:", len(berubah))
print("Teks asli, ID, dan field lainnya tetap.")

Jumlah kalimat: 3140
Kalimat yang berubah karena tanggal: 2
Teks asli, ID, dan field lainnya tetap.


## 6. Simpan

In [ ]:
data["kalimat"] = kalimat

meta = data.setdefault("metadata", {})
meta["jumlah_kalimat"] = len(kalimat)
meta["normalisasi_teks_cari"] = (
    "Huruf kecil, spasi rapi, normalisasi alias, "
    "dan tanggal dalam format d bulan yyyy."
)
meta["jumlah_kalimat_berubah_tanggal"] = len(berubah)

nama_output = "korpus_kalimat_final.json"

Path(nama_output).write_text(
    json.dumps(data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Jumlah kalimat:", len(kalimat))

files.download(nama_output)

Tersimpan: korpus_kalimat_final.json
Jumlah kalimat: 3140


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>